# 01 · Data overview
> Run the notebooks in order (01 → 02 → 03 → 04 → 05 → 06 → 07 → 08) from inside the `code` folder. Raw inputs
> are in `data/`. Research questions: top of notebook 03.


**Project:** What's in a buzzword? Corporate rebranding in the Dot-com, Blockchain and AI waves.

This notebook describes the raw input files of the project and checks their size, date coverage and key
identifiers. All later notebooks read these files; nothing is downloaded while the notebooks run.

| Notebook | Purpose |
|---|---|
| **01_data_overview** (this one) | Raw input files: sources, coverage and identifiers |
| 02_events_and_dates | Name-change events, identifier checks, SEC EDGAR / press dates and the **event register** (verified public dates, confidence grades, deal types) |
| 03_core_analysis | EDA, RQ1–RQ4 (prevalence, event study, firm characteristics, AI vs Dot-com) on the primary sample |
| 04_announcement_extension | **Appendix:** eligible 8-K mentions of blockchain / AI after 24 months of silence (EDGAR full-text search) |
| 05_control_group | Robustness: thematic vs. non-thematic renames, both on the CRSP effective date |
| 06_event_clustering | Appendix check: event clustering in the 8-K sample (clustered SEs, daily calendar-time portfolios) |
| 07_confounding_events | Results by deal type (cosmetic, pivot, merger, de-SPAC, other news) and the sensitivity table |
| 08_micropublication_figures | Figure panels for the micropublication (from the result tables in `output/`) |

All event-study calculations of notebooks 03–07 come from one shared module, `buzzword_event_study.py`, in this folder.

### Raw input files (`data/`)
* `data/raw_data/wrds/`: extracts from WRDS (CRSP, Compustat, CRSP/Compustat Merged, Fama-French factors),
  retrieved on 2026-10-15. The table below lists the WRDS source table of each file.
* `data/raw_data/edgar_fts/`: SEC EDGAR full-text search results for 8-K filings that mention the buzzwords
  (notebook 04).
* `data/event_dates/`: inputs for the event dates (notebook 02): the SEC EDGAR filing-scan results and the
  web-search log (`sec_work/`), and the event-date audit with the public announcement date, confidence grade and
  deal type of every event (`event_audit.csv`).

| File | WRDS table | Content | Level |
|---|---|---|---|
| `crsp_security_info_history.csv.gz` | `crsp_a_stock.stksecurityinfohist` | Exchange, share type, ticker, CUSIP, names over time | security (PERMNO) |
| `crsp_issuer_info_history.csv.gz` | `crsp_a_stock.stkissuerinfohist` | **Company name history** | company (PERMCO) |
| `crsp_daily_market_index.csv.gz` | CRSP daily market indexes | `vwretd`, `ewretd`, `sprtrn` | market |
| `ccm_link_history.csv.gz` | `crsp_a_ccm.ccmxpf_lnkhist` | PERMNO ↔ GVKEY links with validity dates | link |
| `ccm_permno_gvkey_cik.csv.gz` | CCM + `comp.company` | PERMNO ↔ GVKEY ↔ SEC CIK (to find EDGAR filings) | link |
| `compustat_funda_1987_2025.csv.gz` | `comp.funda` + `comp.company` | Annual fundamentals | company (GVKEY) |
| `crsp_daily_event_firms.csv.gz` | `crsp_a_stock.stkdlysecuritydata` | Daily returns and volume of the event firms | security |
| `crsp_daily_announcement_firms.csv.gz` | `crsp_a_stock.stkdlysecuritydata` | Daily data of the 8-K announcement firms (notebook 04) | security |
| `crsp_daily_rename_firms.csv.gz` | `crsp_a_stock.stkdlysecuritydata` | Daily data of every renaming firm 1994–2025, [−400, +30] calendar days around each name change (control group, notebook 05) | security |
| `crsp_monthly_announcement_firms.csv.gz` | `crsp_a_stock.stkmthsecuritydata` | Monthly returns and market cap of the 8-K announcement firms 2010–2025 (calendar-time portfolios, notebook 06) | security |
| `ff_factors_daily.csv.gz`, `ff_factors_monthly.csv.gz` | `ff_all.factors_daily`, `ff_all.factors_monthly` | Fama-French market, SMB, HML, momentum (UMD) and risk-free rate | market |

In [1]:
from pathlib import Path

import pandas as pd

In [2]:
# ---- paths: run the notebooks from inside the code/ folder ----
base_dir = Path.cwd().parent                                  # project folder (the notebooks run from code/)
data_dir = base_dir / "data"                                  # raw input files (described in notebook 01)
wrds_dir = data_dir / "raw_data" / "wrds"                     # WRDS extracts
edgar_dir = data_dir / "raw_data" / "edgar_fts"               # EDGAR full-text search results
event_dates_dir = data_dir / "event_dates"                    # event-date inputs (notebook 02)

In [3]:
for folder in [wrds_dir, edgar_dir, event_dates_dir, event_dates_dir / "sec_work"]:
    print(folder.relative_to(base_dir))
    for file in sorted(p for p in folder.glob("*") if p.is_file() and not p.name.startswith(".")):
        print(f"  {file.name:<45} {file.stat().st_size / 1e6:7.2f} MB")

data/raw_data/wrds
  ccm_link_history.csv.gz                          1.09 MB
  ccm_permno_gvkey_cik.csv.gz                      0.87 MB
  compustat_funda_1987_2025.csv.gz                28.70 MB
  crsp_daily_announcement_firms.csv.gz            11.40 MB
  crsp_daily_event_firms.csv.gz                    3.34 MB
  crsp_daily_market_index.csv.gz                   0.11 MB
  crsp_daily_rename_firms.csv.gz                  47.56 MB
  crsp_issuer_info_history.csv.gz                  2.50 MB
  crsp_monthly_announcement_firms.csv.gz           3.69 MB
  crsp_security_info_history.csv.gz                4.14 MB
  ff_factors_daily.csv.gz                          0.10 MB
  ff_factors_monthly.csv.gz                        0.01 MB
data/raw_data/edgar_fts
  edgar_fts_8k_hits.csv.gz                         0.61 MB
data/event_dates
  event_audit.csv                                  0.01 MB
data/event_dates/sec_work
  other_sources_log.csv                            0.01 MB
  sec_scan_results.json      

### First look at the WRDS extracts
Size, date coverage and key identifiers of each extract, as in Tutorial 4's first look at the data.

In [4]:
checks = {
    "crsp_security_info_history": ("secinfostartdt", "permno"),
    "crsp_issuer_info_history": ("issinfostartdt", "permco"),
    "crsp_daily_market_index": ("dlycaldt", None),
    "ccm_link_history": ("linkdt", "gvkey"),
    "ccm_permno_gvkey_cik": ("linkdt", "permno"),
    "compustat_funda_1987_2025": ("datadate", "gvkey"),
}
rows = []
for name, (date_column, id_column) in checks.items():
    table = pd.read_csv(wrds_dir / f"{name}.csv.gz", low_memory=False)
    rows.append({"file": name, "rows": len(table), "columns": table.shape[1],
                 "first_date": table[date_column].min(), "last_date": table[date_column].max(),
                 "unique_ids": table[id_column].nunique() if id_column else None})
pd.DataFrame(rows)

,file,rows,columns,first_date,last_date,unique_ids
0,crsp_security_info_history,191048,28,1925-12-31,2025-12-31,40518.0
1,crsp_issuer_info_history,139528,18,1925-12-31,2025-12-31,33163.0
2,crsp_daily_market_index,9572,4,1988-01-04,2025-12-31,NaN
3,ccm_link_history,123388,8,1946-01-01,2026-01-30,54704.0
4,ccm_permno_gvkey_cik,33324,6,1949-07-01,2025-12-22,29567.0
5,compustat_funda_1987_2025,444910,27,1987-06-30,2026-05-31,41559.0


### Files for the control group and the calendar-time portfolios
Notebook 05 uses daily data for **all** renaming firms (not only the thematic ones), [−400, +30] calendar days around every name change from 1994 on; notebook 06 uses the Fama-French factors and the monthly returns of the 8-K announcement firms.

In [5]:
rows = []
for name, (date_column, id_column) in {"ff_factors_daily": ("date", None), "ff_factors_monthly": ("date", None),
                                        "crsp_daily_rename_firms": ("dlycaldt", "permno"),
                                        "crsp_monthly_announcement_firms": ("mthcaldt", "permno")}.items():
    table = pd.read_csv(wrds_dir / f"{name}.csv.gz", usecols=[date_column] + ([id_column] if id_column else []))
    rows.append({"file": name, "rows": len(table), "first_date": table[date_column].min(),
                 "last_date": table[date_column].max(), "unique_ids": table[id_column].nunique() if id_column else None})
pd.DataFrame(rows)

,file,rows,first_date,last_date,unique_ids
0,ff_factors_daily,9738,1988-01-04,2026-08-31,NaN
1,ff_factors_monthly,464,1988-01-01,2026-08-01,NaN
2,crsp_daily_rename_firms,2897319,1992-11-30,2025-12-31,4833.0
3,crsp_monthly_announcement_firms,216907,2010-01-29,2025-12-31,1824.0
